# Music Annotation Project Notebook 66
## Human Semantic Review of 59R3 Retained-Region Families

**Runtime:** CPU. Select **Runtime → Run all** in Google Colab.

Notebook 66 turns the diagnostic output from Notebook 59R3 into an authoritative, auditable human-review dataset. It does not retrain, change thresholds, modify corrected pages or promote a model.

### What the review adds

Notebook 59R3 can measure location, darkness, probability, elongation, topology, local context and cluster membership. It cannot determine semantic ownership reliably. The reviewer therefore records:

1. whether the candidate is **ANNOTATION**, **GENUINE_PRINT**, **MIXED_OVERLAP**, **UNCERTAIN**, or **MASK_ERROR**;
2. whether the connected-region boundary represents one coherent mark or a fragmented/merged construction;
3. whether the candidate matches a practical family such as a top-right part/catalogue identifier, long arrow/glyph, dark pen or pale pencil;
4. whether 59R3's diagnostic family flag is correct;
5. whether the region can be judged at 33×33 context, needs approximately 97×97 context, needs whole-page/page-role context, or remains uncertain;
6. reviewer confidence, notes and optional treatment recommendation.

The initial review view is deliberately **blinded to cluster ID, probability and automatic family flags**. Those fields can be revealed after a decision for diagnosis, but they are not needed to decide what the mark actually is.

### Review target

The default queue contains approximately 500 page-diversified regions:

- 100 top-right identifier candidates;
- 100 long-glyph candidates;
- 100 large dark Cluster 1 candidates;
- 100 pale near-threshold Cluster 0 candidates;
- 100 high-context/possible print-overlap candidates.

Duplicates across families are removed. Replacement candidates are drawn from the same family when possible. The selections and all source hashes are persisted before Gradio launches.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q gradio opencv-python-headless pandas numpy pyarrow matplotlib pillow


In [ ]:
# ============================================================
# Cell 2 - Configuration and immutable run identity
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, sys, zipfile

import cv2
import gradio as gr
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
R3_ROOT = ROOT / 'notebook59R3_retained_morphology'
R3_RUN_OVERRIDE = None

EXPECTED_PAGES = 151
TARGET_PER_STRATUM = 100
RANDOM_SEED = 66
PUBLIC_LINK = True

# Context views. 33 and 97 test the model-context hypothesis; 241 gives local score context.
CONTEXT_SIZES = [33, 97, 241]
DEFAULT_CONTEXT = 97
OVERVIEW_MAX_DIMENSION = 1500

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook66_human_semantic_review'
OUT = ROOT / 'notebook66_human_semantic_review' / RUN_ID
INPUTS = OUT / 'inputs'
TABLES = OUT / 'tables'
DECISIONS = OUT / 'decisions'
VISUALS = OUT / 'visuals'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for folder in [OUT, INPUTS, TABLES, DECISIONS, VISUALS, AUDIT, UPLOAD]:
    folder.mkdir(parents=True, exist_ok=False)

print('Notebook 66 output:', OUT)


In [ ]:
# ============================================================
# Cell 3 - Helpers and exact Notebook 59R3 run resolution
# ============================================================

REQUIRED_R3 = [
    'tables/retained_regions_clustered.csv',
    'tables/retained_region_features.csv',
    'tables/retained_region_pixels.parquet',
    'tables/special_family_summary.csv',
    'audit/input_identity.json',
    'audit/notebook59R3_verdict.json',
]

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def atomic_json(path, value):
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def latest_complete(root, required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):
                hits.append(p)
    return max(hits, key=lambda p:p.stat().st_mtime) if hits else None

R3 = Path(R3_RUN_OVERRIDE) if R3_RUN_OVERRIDE else latest_complete(R3_ROOT, REQUIRED_R3)
require(R3 is not None, 'No complete Notebook 59R3 run found. Set R3_RUN_OVERRIDE.')
require(all((R3/r).exists() for r in REQUIRED_R3), f'Incomplete Notebook 59R3 run: {R3}')

r3_verdict = json.loads((R3/'audit'/'notebook59R3_verdict.json').read_text())
require(r3_verdict.get('verdict') == 'PASS', 'Notebook 59R3 verdict is not PASS.')
r3_identity = json.loads((R3/'audit'/'input_identity.json').read_text())

print('Notebook 59R3 source:', R3)
print('59R3 pages:', r3_identity.get('pages'))


In [ ]:
# ============================================================
# Cell 4 - Load region evidence and resolve authoritative images
# ============================================================

features = pd.read_csv(R3/'tables'/'retained_regions_clustered.csv')
pixels = pd.read_parquet(R3/'tables'/'retained_region_pixels.parquet')
require(not features.region_id.duplicated().any(), 'Duplicate region_id values in 59R3 features.')
require(set(pixels.region_id).issubset(set(features.region_id)), 'Region-pixel table contains unknown region IDs.')

# Prefer Notebook 64's exact page resolution referenced by 59R3. Fall back to a path already stored in 59R3 identity.
pair_csv = Path(r3_identity['page_resolution_csv'])
require(pair_csv.exists(), f'59R3 page-resolution CSV no longer exists: {pair_csv}')
resolution = pd.read_csv(pair_csv)
resolution['page'] = resolution.page.astype(str)
observed_col = next((c for c in ['observed_path','observed','aligned_output','resolved_observed_path'] if c in resolution.columns), None)
require(observed_col is not None, f'No observed path column in page resolution: {list(resolution.columns)}')

observed_paths={}
for _,r in resolution.iterrows():
    p=Path(str(r[observed_col]))
    if p.exists(): observed_paths[str(r.page)] = p
missing=sorted(set(features.page.astype(str))-set(observed_paths))
require(not missing, f'Observed pages unresolved for review: {missing[:20]}')

features['page']=features.page.astype(str)
features['region_id']=features.region_id.astype(str)
features['reviewed']=False
print('Candidate regions:', len(features), 'pages:', features.page.nunique())


In [ ]:
# ============================================================
# Cell 5 - Build and lock a page-diversified 500-region queue
# ============================================================

rng = np.random.default_rng(RANDOM_SEED)
selected=[]
used=set()

def diverse_pick(frame, stratum, target=TARGET_PER_STRATUM, ascending_probability=True):
    q=frame.copy()
    if q.empty: return
    # Priority combines practical salience with page diversity. One candidate per page first,
    # then a second pass if the family has fewer than target unique pages.
    q['priority_pixels']=q.pixels.rank(pct=True)
    q['priority_low_p']=(1-q.p_remove_median.rank(pct=True)) if ascending_probability else q.p_remove_median.rank(pct=True)
    q['priority']=0.65*q.priority_pixels + 0.35*q.priority_low_p
    q=q.sort_values('priority',ascending=False)
    passes=[q.drop_duplicates('page'), q]
    count=0
    for pool in passes:
        for _,row in pool.iterrows():
            rid=str(row.region_id)
            if rid in used: continue
            record=row.to_dict(); record['review_stratum']=stratum; record['stratum_rank']=count+1
            selected.append(record); used.add(rid); count+=1
            if count>=target: return

# 1. Top-right identifiers such as D32.
diverse_pick(features[features.flag_top_right_identifier_candidate.astype(bool)], 'TOP_RIGHT_IDENTIFIER', ascending_probability=True)
# 2. Long glyphs/arrows.
diverse_pick(features[features.flag_long_glyph_candidate.astype(bool)], 'LONG_GLYPH_OR_ARROW', ascending_probability=True)
# 3. Large dark regions from Cluster 1.
diverse_pick(features[(features.cluster_id==1) & (features.flag_dark_pen_candidate.astype(bool) | features.flag_dense_dark_candidate.astype(bool))], 'LARGE_DARK_CLUSTER1', ascending_probability=True)
# 4. Pale material close to threshold from Cluster 0.
diverse_pick(features[(features.cluster_id==0) & features.flag_pale_near_threshold.astype(bool)], 'PALE_NEAR_THRESHOLD_CLUSTER0', ascending_probability=False)
# 5. Dense surrounding print context / likely overlap.
context_cut=max(float(features.context_edge_density.quantile(.85)), .08)
diverse_pick(features[(features.context_edge_density>=context_cut) | (features.context_dark_density>=features.context_dark_density.quantile(.90))], 'HIGH_CONTEXT_OR_PRINT_OVERLAP', ascending_probability=True)

queue=pd.DataFrame(selected)
require(len(queue)>0, 'Review queue is empty.')
queue=queue.sort_values(['review_stratum','stratum_rank','page','region_id']).reset_index(drop=True)
queue['queue_index']=np.arange(len(queue),dtype=int)
queue['review_uid']=queue.region_id.map(lambda x:'N66:'+str(x))
queue['review_status']='UNREVIEWED'
queue['selected_utc']=datetime.now(timezone.utc).isoformat()

# Lock the queue before review.
queue_path=INPUTS/'review_queue_locked.csv'
queue.to_csv(queue_path,index=False)
shutil.copy2(R3/'tables'/'retained_region_pixels.parquet', INPUTS/'retained_region_pixels_source.parquet')

queue_summary=queue.groupby('review_stratum',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'))
queue_summary.to_csv(TABLES/'review_queue_summary.csv',index=False)

lock={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_59R3_run':str(R3),
    'source_59R3_verdict_sha256':sha256_file(R3/'audit'/'notebook59R3_verdict.json'),
    'source_features_sha256':sha256_file(R3/'tables'/'retained_regions_clustered.csv'),
    'source_pixels_sha256':sha256_file(R3/'tables'/'retained_region_pixels.parquet'),
    'queue_sha256':sha256_file(queue_path),
    'random_seed':RANDOM_SEED,
    'target_per_stratum':TARGET_PER_STRATUM,
    'rows':len(queue),
    'pages':queue.page.nunique(),
    'duplicate_regions':int(queue.region_id.duplicated().sum()),
}
atomic_json(INPUTS/'review_queue_lock.json',lock)
require(lock['duplicate_regions']==0, 'Review queue contains duplicate regions.')
print(queue_summary.to_string(index=False))
print('Locked review rows:',len(queue),'across',queue.page.nunique(),'pages')


In [ ]:
# ============================================================
# Cell 6 - Review state and durable decision schema
# ============================================================

OWNERSHIP = ['ANNOTATION','GENUINE_PRINT','MIXED_OVERLAP','UNCERTAIN','MASK_ERROR']
COHERENCE = ['ONE_COHERENT_MARK','FRAGMENT_OF_LARGER_MARK','MULTIPLE_MERGED_MARKS','BOUNDARY_INCORRECT','NOT_APPLICABLE','UNCERTAIN']
HUMAN_FAMILY = ['TOP_RIGHT_PART_OR_CATALOGUE_ID','LONG_GLYPH_OR_ARROW','DARK_PEN_HANDWRITING','PALE_PENCIL_HANDWRITING','PRINT_OVERLAP','OTHER_ANNOTATION','GENUINE_PRINT_STRUCTURE','MASK_ERROR','UNCERTAIN']
CONTEXT_REQUIRED = ['33PX_SUFFICIENT','97PX_REQUIRED','241PX_REQUIRED','WHOLE_PAGE_OR_PAGE_ROLE_REQUIRED','STILL_UNCERTAIN']
TREATMENT = ['CALIBRATION_OR_THRESHOLD_RESEARCH','DARK_INK_SPATIAL_CHALLENGER','LARGER_OR_MULTISCALE_CONTEXT','PAGE_ROLE_LAYOUT_ABLATION','CONSERVATIVE_UNCERTAIN_FALLBACK','MASK_REVIEW','NO_MODEL_CHANGE','OTHER_OR_UNCERTAIN']

DECISION_COLUMNS=[
    'review_uid','region_id','queue_index','page','source_group','review_stratum',
    'ownership_label','coherence_label','human_family','context_required','treatment_recommendation',
    'reviewer_confidence','flag_match','first_page_identifier_visible','long_glyph_visible',
    'notes','reviewer','reviewed_utc','revision','source_queue_sha256'
]

state_path=DECISIONS/'human_review_decisions.csv'
history_path=DECISIONS/'human_review_history.jsonl'

def empty_decisions():
    return pd.DataFrame(columns=DECISION_COLUMNS)

def load_decisions():
    if not state_path.exists(): return empty_decisions()
    d=pd.read_csv(state_path,dtype=str).fillna('')
    missing=set(DECISION_COLUMNS)-set(d.columns)
    require(not missing, f'Decision file missing columns: {sorted(missing)}')
    return d[DECISION_COLUMNS].copy()

def persist_decisions(df):
    tmp=Path(str(state_path)+'.tmp')
    df.to_csv(tmp,index=False)
    tmp.replace(state_path)

persist_decisions(empty_decisions())
print('Decision schema ready:',state_path)


In [ ]:
# ============================================================
# Cell 7 - Visual construction: blinded first, diagnostics optional
# ============================================================

image_cache={}
pixel_groups={rid:g.copy() for rid,g in pixels.groupby('region_id',sort=False)}

def get_observed(page):
    if page not in image_cache:
        im=cv2.imread(str(observed_paths[page]),cv2.IMREAD_GRAYSCALE)
        require(im is not None, f'Unreadable observed page: {observed_paths[page]}')
        image_cache[page]=im
    return image_cache[page]

def context_bounds(row,size):
    im=get_observed(row.page); H,W=im.shape
    cx=int(round(row.cx)); cy=int(round(row.cy)); half=size//2
    x0=max(0,cx-half); y0=max(0,cy-half); x1=min(W,cx+half+1); y1=min(H,cy+half+1)
    # preserve requested size where possible by extending opposite edge
    if x1-x0<size: x0=max(0,x1-size); x1=min(W,x0+size)
    if y1-y0<size: y0=max(0,y1-size); y1=min(H,y0+size)
    return x0,y0,x1,y1

def region_crop(row,size,show_box=True,show_region=True):
    im=get_observed(row.page); x0,y0,x1,y1=context_bounds(row,size)
    crop=cv2.cvtColor(im[y0:y1,x0:x1],cv2.COLOR_GRAY2RGB)
    if show_region:
        q=pixel_groups.get(row.region_id)
        if q is not None:
            yy=q.y.to_numpy(int)-y0; xx=q.x.to_numpy(int)-x0
            valid=(yy>=0)&(xx>=0)&(yy<crop.shape[0])&(xx<crop.shape[1])
            overlay=crop.copy(); overlay[yy[valid],xx[valid]]=(255,0,255)
            crop=cv2.addWeighted(crop,.62,overlay,.38,0)
    if show_box:
        cv2.rectangle(crop,(max(0,int(row.x)-x0),max(0,int(row.y)-y0)),(min(crop.shape[1]-1,int(row.x+row.w)-x0),min(crop.shape[0]-1,int(row.y+row.h)-y0)),(255,0,0),1)
    return crop

def whole_page_locator(row):
    im=get_observed(row.page); rgb=cv2.cvtColor(im,cv2.COLOR_GRAY2RGB)
    cv2.rectangle(rgb,(int(row.x),int(row.y)),(int(row.x+row.w),int(row.y+row.h)),(255,0,0),5)
    scale=min(1.0,OVERVIEW_MAX_DIMENSION/max(rgb.shape[:2]))
    if scale<1: rgb=cv2.resize(rgb,None,fx=scale,fy=scale,interpolation=cv2.INTER_AREA)
    return rgb

def blinded_header(row):
    return f'Queue {int(row.queue_index)+1}/{len(queue)} | Page {row.page} | Stratum {row.review_stratum}'

def diagnostic_text(row):
    return (
        f"Region {row.region_id}
"
        f"Cluster {int(row.cluster_id)} | band {row.band_id} | pixels {int(row.pixels)}
"
        f"Observed median {row.observed_median:.1f} | p_remove median {row.p_remove_median:.4f}
"
        f"Aspect {row.aspect_ratio:.2f} | linearity {row.linearity:.3f} | fill {row.bbox_fill:.3f}
"
        f"59R3 flags: top-right={bool(row.flag_top_right_identifier_candidate)}, long={bool(row.flag_long_glyph_candidate)}, "
        f"pale={bool(row.flag_pale_near_threshold)}, dark-pen={bool(row.flag_dark_pen_candidate)}, dense-dark={bool(row.flag_dense_dark_candidate)}"
    )


In [ ]:
# ============================================================
# Cell 8 - Decision, navigation and summary callbacks
# ============================================================

def row_at(index):
    index=max(0,min(int(index),len(queue)-1))
    return queue.iloc[index]

def existing_for(rid):
    d=load_decisions(); q=d[d.region_id==str(rid)]
    return None if q.empty else q.iloc[-1]

def render(index,context_size=DEFAULT_CONTEXT,reveal=False):
    row=row_at(index); existing=existing_for(row.region_id)
    status='UNREVIEWED' if existing is None else f"REVIEWED: {existing.ownership_label} | confidence {existing.reviewer_confidence}%"
    diag=diagnostic_text(row) if reveal else 'Diagnostic features hidden until requested.'
    values=['','','','','',50,'','','', '', '']
    if existing is not None:
        values=[existing.ownership_label,existing.coherence_label,existing.human_family,existing.context_required,existing.treatment_recommendation,float(existing.reviewer_confidence or 50),existing.flag_match,existing.first_page_identifier_visible,existing.long_glyph_visible,existing.notes,existing.reviewer]
    return (
        int(row.queue_index), blinded_header(row), status,
        region_crop(row,33),region_crop(row,97),region_crop(row,241),whole_page_locator(row),diag,
        *values
    )

def save_decision(index,ownership,coherence,family,context,treatment,confidence,flag_match,first_page,long_glyph,notes,reviewer):
    row=row_at(index)
    require(ownership in OWNERSHIP, 'Choose an ownership label.')
    require(coherence in COHERENCE, 'Choose a coherence label.')
    require(family in HUMAN_FAMILY, 'Choose a human family.')
    require(context in CONTEXT_REQUIRED, 'Choose the minimum context required.')
    require(treatment in TREATMENT, 'Choose a treatment recommendation.')
    reviewer=str(reviewer).strip() or 'David Bolton'
    d=load_decisions(); old=d[d.region_id==row.region_id]
    revision=1 if old.empty else int(pd.to_numeric(old.revision,errors='coerce').max())+1
    record={
        'review_uid':row.review_uid,'region_id':row.region_id,'queue_index':int(row.queue_index),'page':row.page,'source_group':row.source_group,'review_stratum':row.review_stratum,
        'ownership_label':ownership,'coherence_label':coherence,'human_family':family,'context_required':context,'treatment_recommendation':treatment,
        'reviewer_confidence':int(confidence),'flag_match':flag_match,'first_page_identifier_visible':first_page,'long_glyph_visible':long_glyph,
        'notes':str(notes),'reviewer':reviewer,'reviewed_utc':datetime.now(timezone.utc).isoformat(),'revision':revision,'source_queue_sha256':sha256_file(INPUTS/'review_queue_locked.csv')
    }
    # Current decision table contains one row per region; history remains append-only.
    d=d[d.region_id!=row.region_id]
    d=pd.concat([d,pd.DataFrame([record])],ignore_index=True)
    persist_decisions(d)
    with open(history_path,'a') as f: f.write(json.dumps(record,default=str)+'
')
    return f'SAVED {row.region_id}: {ownership} / {family} / confidence {int(confidence)}%'

def save_and_next(*args):
    message=save_decision(*args)
    index=min(int(args[0])+1,len(queue)-1)
    return (message,*render(index,DEFAULT_CONTEXT,False))

def jump_unreviewed():
    done=set(load_decisions().region_id.astype(str))
    for _,row in queue.iterrows():
        if row.region_id not in done: return render(int(row.queue_index),DEFAULT_CONTEXT,False)
    return render(len(queue)-1,DEFAULT_CONTEXT,False)

def progress_table():
    d=load_decisions()
    reviewed=set(d.region_id.astype(str))
    q=queue.copy(); q['complete']=q.region_id.isin(reviewed)
    s=q.groupby('review_stratum',as_index=False).agg(total=('region_id','size'),reviewed=('complete','sum'),pages=('page','nunique'))
    s['remaining']=s.total-s.reviewed
    return s

def label_summary():
    d=load_decisions()
    if d.empty:return pd.DataFrame(columns=['ownership_label','regions','pages'])
    return d.groupby('ownership_label',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))


In [ ]:
# ============================================================
# Cell 9 - Launch the human semantic reviewer
# ============================================================

with gr.Blocks(title='Notebook 66 - 59R3 Human Semantic Review') as app:
    gr.Markdown('# Notebook 66: Human semantic review of 59R3 retained regions')
    gr.Markdown('Decide semantic ownership from the imagery first. Reveal 59R3 diagnostics only after forming an independent judgement.')

    with gr.Row():
        index=gr.Number(value=0,precision=0,label='Queue index (0-based)')
        load_btn=gr.Button('Load index')
        next_unreviewed=gr.Button('Jump to next unreviewed',variant='primary')
        reveal=gr.Checkbox(False,label='Reveal 59R3 diagnostics')
    header=gr.Textbox(label='Blinded case identity',interactive=False)
    review_status=gr.Textbox(label='Existing review status',interactive=False)

    with gr.Tab('Blinded semantic review'):
        with gr.Row():
            crop33=gr.Image(label='33x33 context',height=330)
            crop97=gr.Image(label='97x97 context',height=330)
            crop241=gr.Image(label='241x241 context',height=330)
        overview=gr.Image(label='Whole-page locator',height=720)
        diagnostics=gr.Textbox(label='59R3 diagnostics (hidden by default)',interactive=False,lines=7)

    with gr.Tab('Decision form'):
        ownership=gr.Radio(OWNERSHIP,label='1. Semantic ownership')
        coherence=gr.Radio(COHERENCE,label='2. Connected-region coherence')
        family=gr.Dropdown(HUMAN_FAMILY,label='3. Human-observed practical family')
        context=gr.Radio(CONTEXT_REQUIRED,label='4. Minimum context needed to judge this case')
        treatment=gr.Dropdown(TREATMENT,label='5. Treatment hypothesis')
        confidence=gr.Slider(0,100,value=50,step=5,label='Reviewer confidence (%)')
        flag_match=gr.Radio(['YES','PARTLY','NO','CANNOT_TELL'],label='6. Does the 59R3 stratum/flag describe the visible feature?')
        first_page=gr.Radio(['YES','NO','CANNOT_TELL'],label='7. Visible first-page part/catalogue identifier?')
        long_glyph=gr.Radio(['YES','NO','CANNOT_TELL'],label='8. Visible long glyph, arrow or sweeping stroke?')
        notes=gr.Textbox(label='Notes',lines=4)
        reviewer=gr.Textbox(value='David Bolton',label='Reviewer')
        with gr.Row():
            save_btn=gr.Button('Save decision')
            save_next_btn=gr.Button('Save and next',variant='primary')
        save_status=gr.Textbox(label='Save status',interactive=False)

    with gr.Tab('Progress and current labels'):
        refresh_progress=gr.Button('Refresh progress')
        progress=gr.Dataframe(label='Progress by review stratum',interactive=False)
        labels=gr.Dataframe(label='Ownership labels collected',interactive=False)

    render_outputs=[index,header,review_status,crop33,crop97,crop241,overview,diagnostics,ownership,coherence,family,context,treatment,confidence,flag_match,first_page,long_glyph,notes,reviewer]
    load_btn.click(render,[index,gr.State(DEFAULT_CONTEXT),reveal],render_outputs)
    index.change(render,[index,gr.State(DEFAULT_CONTEXT),reveal],render_outputs)
    reveal.change(render,[index,gr.State(DEFAULT_CONTEXT),reveal],render_outputs)
    next_unreviewed.click(jump_unreviewed,outputs=render_outputs)

    decision_inputs=[index,ownership,coherence,family,context,treatment,confidence,flag_match,first_page,long_glyph,notes,reviewer]
    save_btn.click(save_decision,decision_inputs,save_status)
    save_next_btn.click(save_and_next,decision_inputs,[save_status,*render_outputs])
    refresh_progress.click(lambda:(progress_table(),label_summary()),outputs=[progress,labels])
    app.load(render,[gr.State(0),gr.State(DEFAULT_CONTEXT),gr.State(False)],render_outputs)
    app.load(lambda:(progress_table(),label_summary()),outputs=[progress,labels])

app.launch(share=PUBLIC_LINK,debug=False,show_error=True)


In [ ]:
# ============================================================
# Cell 10 - Run after review: completeness, agreement and purity tables
# ============================================================

# Stop the Gradio cell above when finished, then run this cell.
decisions=load_decisions()
reviewed=queue.merge(decisions,on=['review_uid','region_id','queue_index','page','source_group','review_stratum'],how='left',validate='one_to_one',suffixes=('','_review'))
reviewed['is_reviewed']=reviewed.ownership_label.notna() & (reviewed.ownership_label!='')
reviewed.to_csv(TABLES/'review_queue_with_decisions.csv',index=False)

completeness=reviewed.groupby('review_stratum',as_index=False).agg(
    selected=('region_id','size'),reviewed=('is_reviewed','sum'),pages=('page','nunique')
)
completeness['remaining']=completeness.selected-completeness.reviewed
completeness['completion_fraction']=completeness.reviewed/completeness.selected
completeness.to_csv(TABLES/'review_completeness.csv',index=False)

labelled=reviewed[reviewed.is_reviewed].copy()
if len(labelled):
    ownership_summary=labelled.groupby(['review_stratum','ownership_label'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
    totals=labelled.groupby('review_stratum',as_index=False).size().rename(columns={'size':'reviewed_total'})
    ownership_summary=ownership_summary.merge(totals,on='review_stratum')
    ownership_summary['fraction']=ownership_summary.regions/ownership_summary.reviewed_total
    ownership_summary.to_csv(TABLES/'ownership_purity_by_stratum.csv',index=False)

    cluster_summary=labelled.groupby(['cluster_id','ownership_label'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
    cluster_summary.to_csv(TABLES/'ownership_by_cluster.csv',index=False)

    family_summary=labelled.groupby(['human_family','ownership_label'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),median_confidence=('reviewer_confidence','median'))
    family_summary.to_csv(TABLES/'ownership_by_human_family.csv',index=False)

    context_summary=labelled.groupby(['human_family','context_required'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
    context_summary.to_csv(TABLES/'context_requirement_by_family.csv',index=False)

    treatment_summary=labelled.groupby(['human_family','treatment_recommendation'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
    treatment_summary.to_csv(TABLES/'treatment_hypothesis_by_family.csv',index=False)

    flag_summary=labelled.groupby(['review_stratum','flag_match'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
    flag_summary.to_csv(TABLES/'59R3_flag_validation.csv',index=False)

print(completeness.to_string(index=False))


In [ ]:
# ============================================================
# Cell 11 - Decision-oriented conclusion and review verdict
# ============================================================

require(len(labelled)>0, 'No completed human reviews. Complete at least one review before finalising.')

# Purity is semantic annotation or mixed overlap, depending on the intended family audit.
labelled['annotation_related']=labelled.ownership_label.isin(['ANNOTATION','MIXED_OVERLAP'])
stratum_result=labelled.groupby('review_stratum',as_index=False).agg(
    reviewed=('region_id','size'),pages=('page','nunique'),
    annotation_related=('annotation_related','sum'),
    median_confidence=('reviewer_confidence','median'),
    flag_yes=('flag_match',lambda x:int((x=='YES').sum())),
    flag_no=('flag_match',lambda x:int((x=='NO').sum()))
)
stratum_result['annotation_related_fraction']=stratum_result.annotation_related/stratum_result.reviewed
stratum_result['flag_yes_fraction']=stratum_result.flag_yes/stratum_result.reviewed
stratum_result.to_csv(TABLES/'human_validation_by_stratum.csv',index=False)

# Context evidence directly answers whether a larger field of view is needed.
context_result=labelled.groupby('context_required',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
context_result['fraction']=context_result.regions/len(labelled)
context_result.to_csv(TABLES/'context_requirement_summary.csv',index=False)

# Fail closed on an incomplete review when declaring the definitive experiment result.
completion=float(labelled.region_id.nunique()/len(queue))
definitive=completion>=0.95
result={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'source_59R3_run':str(R3),
    'queue_rows':int(len(queue)),
    'reviewed_rows':int(labelled.region_id.nunique()),
    'completion_fraction':completion,
    'definitive_review_complete':definitive,
    'semantic_truth_added':True,
    'clean_truth_used':False,
    'model_or_threshold_changed':False,
    'promotion_authority':False,
    'interpretation':('Human semantic review is sufficiently complete for family-purity and context analysis.' if definitive else 'Partial human review only; preserve as interim evidence and do not make definitive purity claims.'),
}
atomic_json(OUT/'notebook66_review_result.json',result)
print(json.dumps(result,indent=2))
print('
Human validation by stratum:')
print(stratum_result.to_string(index=False))
print('
Context requirement:')
print(context_result.to_string(index=False))


In [ ]:
# ============================================================
# Cell 12 - Final upload package
# ============================================================

checks={
    'source_59R3_passed':r3_verdict.get('verdict')=='PASS',
    'queue_locked':(INPUTS/'review_queue_lock.json').is_file(),
    'queue_unique':not queue.region_id.duplicated().any(),
    'decision_table_exists':state_path.is_file(),
    'review_result_exists':(OUT/'notebook66_review_result.json').is_file(),
    'review_rows_exist':len(labelled)>0,
    'source_queue_hash_preserved':bool((labelled.source_queue_sha256==sha256_file(INPUTS/'review_queue_locked.csv')).all()),
    'no_clean_truth_used':True,
    'no_model_or_threshold_change':True,
}
first_failure=next((k for k,v in checks.items() if not v),None)
verdict='PASS' if first_failure is None else 'FAIL'
verdict_record={'verdict':verdict,'first_failed_check':first_failure,'checks':checks,'completion_fraction':float(len(labelled)/len(queue)),'created_utc':datetime.now(timezone.utc).isoformat(),'promotion_authority':False}
atomic_json(AUDIT/'notebook66_verdict.json',verdict_record)
require(verdict=='PASS',f'Notebook 66 packaging blocked: {first_failure}')

summary_lines=[
    'NOTEBOOK 66 - HUMAN SEMANTIC REVIEW OF 59R3 RETAINED REGIONS','',
    f'Verdict: {verdict}',f'59R3 source: {R3}',f'Queue: {len(queue)} regions across {queue.page.nunique()} pages',
    f'Reviewed: {len(labelled)} ({len(labelled)/len(queue):.1%})','',
    'WHAT HUMAN REVIEW ADDS',
    '- semantic ownership rather than morphology alone',
    '- confirmation or rejection of the automatic family flag',
    '- mixed/overlap and mask-error identification',
    '- connected-region coherence and fragmentation evidence',
    '- minimum context needed: 33, 97, 241 or whole-page/page-role',
    '- family-specific treatment recommendation and reviewer confidence','',
    'This remains annotated-only diagnostic evidence. It does not provide clean-page truth, change the model, move a threshold or authorise promotion.'
]
(OUT/'executive_summary.txt').write_text('
'.join(summary_lines)+'
')

curated=[
    OUT/'executive_summary.txt',OUT/'notebook66_review_result.json',AUDIT/'notebook66_verdict.json',
    INPUTS/'review_queue_locked.csv',INPUTS/'review_queue_lock.json',
    TABLES/'review_queue_summary.csv',TABLES/'review_completeness.csv',TABLES/'review_queue_with_decisions.csv',
    TABLES/'human_validation_by_stratum.csv',TABLES/'context_requirement_summary.csv',
    DECISIONS/'human_review_decisions.csv',
]
for optional in [
    TABLES/'ownership_purity_by_stratum.csv',TABLES/'ownership_by_cluster.csv',TABLES/'ownership_by_human_family.csv',
    TABLES/'context_requirement_by_family.csv',TABLES/'treatment_hypothesis_by_family.csv',TABLES/'59R3_flag_validation.csv',
    DECISIONS/'human_review_history.jsonl'
]:
    if optional.exists(): curated.append(optional)

for src in curated:
    require(src.is_file(),f'Missing curated artefact: {src}')
    rel=src.relative_to(OUT); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
    require(sha256_file(src)==sha256_file(dst),f'Copy hash mismatch: {rel}')

(UPLOAD/'README.txt').write_text('Notebook 66 curated human-review package. Semantic labels are human decisions on annotated-only imagery; no clean truth was used.
')
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file() and p.name!='upload_manifest.csv':
        manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('
'.join(x['relative_path'] for x in manifest)+'
')
zip_path=OUT/'Notebook66_Human_Semantic_Review_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(UPLOAD))
with zipfile.ZipFile(zip_path,'r') as z: require(z.testzip() is None,'ZIP integrity test failed.')
atomic_json(OUT/'upload_package_identity.json',{'zip_path':str(zip_path),'zip_sha256':sha256_file(zip_path),'zip_bytes':zip_path.stat().st_size,'files':len(manifest)})
print('
'.join(summary_lines))
print('Upload ZIP:',zip_path)


## Interpretation guide

The principal result is not whether Notebook 59R3 can find dark or elongated shapes. Notebook 59R3 already does that. The principal result is the **semantic purity and treatment relevance** of those selections:

- If top-right candidates are mostly human-confirmed part/catalogue identifiers and require whole-page/page-role context, commission a separately controlled layout-context ablation.
- If long-glyph candidates are mostly annotation and frequently require 97×97 or 241×241 context, commission one multi-scale representation experiment while keeping labels, folds, calibration and decision rules fixed.
- If pale near-threshold candidates are semantically clean and judged adequately at 33×33, calibration/policy may be the main issue.
- If large dark Cluster 1 candidates are mostly dark handwriting with low 59R2 removal probability, add authoritative dark-ink evidence to a spatial challenger.
- If a large fraction is genuine print or mixed overlap, retain conservative UNCERTAIN handling rather than treating the family as removable annotation.
- If 59R3 regions are frequently fragments or merged constructions, region-building must be revised before region-level training or policy analysis.
